# Volatility Targeting Strategy

This notebook implements and analyses a volatility-targeting overlay on SPY over a 20-year sample (2005-present). The strategy rescales the daily position weight so that ex-ante portfolio risk stays constant at 15% annualised, using one-day-lagged realised volatility as the signal. All estimation and simulation logic lives in `src/`; the notebook provides narrative, charts, and verification.


In [ ]:
%matplotlib inline
import sys
import warnings
import logging

sys.path.insert(0, '.')
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data import load_ohlcv
from src.volatility import rolling_std_vol, ewma_vol, garman_klass_vol, yang_zhang_vol
from src.sizing import compute_weights
from src.backtest import run_backtest
from src.metrics import build_summary_table, drawdown_series, sharpe_ratio

plt.rcParams.update({
    'figure.figsize': (13, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
})


In [ ]:
# Strategy parameters -- all downstream code reads from these constants
TICKER       = 'SPY'
START_DATE   = '2005-01-01'
TARGET_VOL   = 0.15
MAX_LEVERAGE = 2.0
ROLL_SHORT   = 20
ROLL_LONG    = 60
EWMA_LAMBDA  = 0.94
COST_BPS     = 1.5

CRISES = [
    ('GFC',        pd.Timestamp('2008-01-01'), pd.Timestamp('2009-06-30')),
    ('COVID-19',   pd.Timestamp('2020-02-01'), pd.Timestamp('2020-12-31')),
    ('Rate Hikes', pd.Timestamp('2022-01-01'), pd.Timestamp('2022-12-31')),
]


## 1. Data Pipeline

We download 20 years of SPY daily OHLCV using split- and dividend-adjusted prices (`auto_adjust=True`). The first run fetches from Yahoo Finance; subsequent runs read from a local parquet cache in `data_cache/`.


In [ ]:
df = load_ohlcv(TICKER, start=START_DATE)
returns = df['log_return'].dropna()

print(f'Period  : {df.index[0].date()} to {df.index[-1].date()}')
print(f'Bars    : {len(df):,} trading days ({(df.index[-1] - df.index[0]).days / 365.25:.1f} years)')
print(f'Columns : {list(df.columns)}')
df.tail(3)


In [ ]:
ann_ret = np.exp(returns.mean() * 252) - 1
ann_vol = returns.std(ddof=1) * np.sqrt(252)

print(f'Buy-and-hold annualised return : {ann_ret:.2%}')
print(f'Buy-and-hold annualised vol    : {ann_vol:.2%}')
print(f'Skewness                       : {returns.skew():.3f}')
print(f'Excess kurtosis                : {returns.kurtosis():.3f}')
print(f'Worst single day               : {returns.min():.2%}  ({returns.idxmin().date()})')
print(f'Best single day                : {returns.max():.2%}  ({returns.idxmax().date()})')


The worst single day (typically 2020-03-16 or 2008-10-15) is a real market event, not a data error. We preserve it — clipping extreme moves would produce a downward-biased volatility surface and overstate the strategy's risk-reduction benefit.


## 2. Volatility Estimators

We compute four estimators at a common 20-day lookback to compare their behaviour during market stress. The 60-day rolling std is added to illustrate the smoothing-responsiveness trade-off inherent in window selection. Crisis windows are shaded (GFC 2008-09, COVID-19 2020, Fed tightening 2022).


In [ ]:
vol_r20  = rolling_std_vol(returns, window=ROLL_SHORT)
vol_r60  = rolling_std_vol(returns, window=ROLL_LONG)
vol_ewma = ewma_vol(returns, lambda_=EWMA_LAMBDA)
vol_gk   = garman_klass_vol(df, window=ROLL_SHORT)
vol_yz   = yang_zhang_vol(df, window=ROLL_SHORT)

# Summary statistics across the full sample
vol_df = pd.DataFrame({
    'Roll-20d'  : vol_r20,
    'Roll-60d'  : vol_r60,
    'EWMA-0.94' : vol_ewma,
    'GK-20d'    : vol_gk,
    'YZ-20d'    : vol_yz,
})
vol_df.describe().loc[['mean', 'std', 'min', 'max']].mul(100).round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for _, s, e in CRISES:
    ax.axvspan(s, e, alpha=0.08, color='crimson')

ax.plot(vol_r20,  lw=1.2, label=f'Roll {ROLL_SHORT}d')
ax.plot(vol_r60,  lw=1.2, label=f'Roll {ROLL_LONG}d')
ax.plot(vol_ewma, lw=1.2, label=f'EWMA lambda={EWMA_LAMBDA}')
ax.plot(vol_yz,   lw=1.2, label='Yang-Zhang 20d')
ax.axhline(TARGET_VOL, color='black', ls='--', lw=1.0, label='15% target')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylabel('Annualised Realised Volatility')
ax.set_title(f'Volatility Estimators -- {TICKER}  (shaded: GFC / COVID-19 / 2022 rate hike cycle)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


All four estimators agree on the direction and approximate magnitude of volatility regimes. The key differences:

- **EWMA** reacts fastest to sudden spikes (COVID Feb 2020: near-instant jump to 80%+), which triggers the most aggressive deleveraging but also the fastest re-leveraging in the recovery. The effective half-life at lambda=0.94 is about 11 days.
- **20-day rolling** is a compromise: it responds within 3-4 weeks and is the most common choice in practice for its interpretability.
- **60-day rolling** lags the spike by several weeks, keeping more exposure during the initial shock and shedding it during the recovery -- often the worst of both worlds for drawdown control.
- **Yang-Zhang** is slightly lower than the rolling std on average because it uses the full OHLC range, which extracts more information per day. The differences are small (< 1%) for a deep ETF like SPY where overnight gaps are minor.

The 20-day rolling std is used as the primary estimator for the backtest.


## 3. Position Sizing

The weight for day *t* is defined as:

    w_t = sigma_target / sigma_{t-1}

capped at 2x and floored at 0 (long-only). The **one-day lag** is mandatory: volatility estimated from data through day t-1 is the only information available at the open of day t. Any implementation that uses same-day volatility has look-ahead bias.


In [ ]:
weights_r20  = compute_weights(vol_r20,  target_vol=TARGET_VOL, max_leverage=MAX_LEVERAGE)
weights_r60  = compute_weights(vol_r60,  target_vol=TARGET_VOL, max_leverage=MAX_LEVERAGE)
weights_ewma = compute_weights(vol_ewma, target_vol=TARGET_VOL, max_leverage=MAX_LEVERAGE)
weights_yz   = compute_weights(vol_yz,   target_vol=TARGET_VOL, max_leverage=MAX_LEVERAGE)

w_stats = pd.DataFrame({
    'Roll-20d'  : weights_r20,
    'Roll-60d'  : weights_r60,
    'EWMA'      : weights_ewma,
    'Yang-Zhang': weights_yz,
}).describe().loc[['mean', 'std', 'min', 'max']].round(3)

print('Weight statistics (post-warmup):')
print(w_stats)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for _, s, e in CRISES:
    ax.axvspan(s, e, alpha=0.08, color='crimson')
ax.plot(weights_r20, lw=0.8, color='steelblue')
ax.axhline(1.0,         color='green', ls='--', lw=1.0, label='Fully invested (1x)')
ax.axhline(MAX_LEVERAGE, color='red',   ls='--', lw=1.0, label=f'Leverage cap ({MAX_LEVERAGE}x)')
ax.set_ylim(bottom=0)
ax.set_ylabel('Weight')
ax.set_title('Daily Position Weight -- Roll 20d')
ax.legend(fontsize=9)

ax = axes[1]
w_valid = weights_r20.dropna()
ax.hist(w_valid, bins=40, color='steelblue', edgecolor='white', lw=0.3)
ax.axvline(w_valid.mean(), color='black', ls='--', lw=1.5,
           label=f'Mean = {w_valid.mean():.2f}x')
ax.set_xlabel('Weight')
ax.set_title('Weight Distribution -- Roll 20d')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


The mean weight is typically 0.8-1.1x depending on the sample period. The leverage cap binds during low-vol regimes (e.g. 2012-2019) when the 15% target exceeds realised vol, pushing the raw weight above 2x. In stress periods, the weight collapses toward zero, shedding exposure precisely when it is most expensive to carry.


## 4. Backtest

We simulate all four vol estimators alongside buy-and-hold. Transaction costs of 1.5 bps per unit of absolute weight change are deducted daily. Both strategies are evaluated from the same start date (the end of the longest warmup period) so the equity curves are directly comparable.


In [ ]:
bt_r20  = run_backtest(df['log_return'], weights_r20,  cost_bps=COST_BPS)
bt_r60  = run_backtest(df['log_return'], weights_r60,  cost_bps=COST_BPS)
bt_ewma = run_backtest(df['log_return'], weights_ewma, cost_bps=COST_BPS)
bt_yz   = run_backtest(df['log_return'], weights_yz,   cost_bps=COST_BPS)

# Align all series to the latest warmup end (60-day rolling needs the longest warmup)
common_start = max(bt_r20.index[0], bt_r60.index[0], bt_ewma.index[0], bt_yz.index[0])
print(f'Common backtest start : {common_start.date()}')
print(f'Backtest length       : {len(bt_r20.loc[common_start:]):,} trading days')
print()
bt_r20[['weight', 'turnover', 'cost', 'strat_net_return', 'bah_return']].head(5)


## 5. Results


In [ ]:
# ── Chart 1: Equity curves (log scale) ───────────────────────────────────────
def _norm(bt, col='strat_equity'):
    s = bt[col].loc[common_start:]
    return s / s.iloc[0]

fig, ax = plt.subplots(figsize=(13, 6))

for _, s, e in CRISES:
    ax.axvspan(s, e, alpha=0.08, color='crimson')

ax.semilogy(_norm(bt_r20),               label='Vol Target - Roll 20d',    lw=1.5)
ax.semilogy(_norm(bt_r60),               label='Vol Target - Roll 60d',    lw=1.5)
ax.semilogy(_norm(bt_ewma),              label='Vol Target - EWMA',        lw=1.5)
ax.semilogy(_norm(bt_yz),                label='Vol Target - Yang-Zhang',  lw=1.5)
ax.semilogy(_norm(bt_r20, 'bah_equity'), label='Buy-and-Hold',             lw=2.0, ls='--', color='black')

ax.axhline(1.0, color='gray', ls=':', lw=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}x'))
ax.set_ylabel('Portfolio Value (log scale, normalised to 1.0)')
ax.set_title(f'Volatility Targeting vs Buy-and-Hold -- {TICKER} (Log Scale)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Chart 2: Rolling realised vol of both strategies ─────────────────────────
strat_rv = rolling_std_vol(bt_r20.loc[common_start:, 'strat_net_return'], window=60)
bah_rv   = rolling_std_vol(bt_r20.loc[common_start:, 'bah_return'],       window=60)

fig, ax = plt.subplots(figsize=(13, 5))
for _, s, e in CRISES:
    ax.axvspan(s, e, alpha=0.08, color='crimson')

ax.plot(strat_rv, lw=1.5, label='Vol-Targeted (Roll 20d)')
ax.plot(bah_rv,   lw=1.5, ls='--', label='Buy-and-Hold')
ax.axhline(TARGET_VOL, color='black', ls='--', lw=1.0, label=f'Target {TARGET_VOL:.0%}')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylabel('Realised Volatility (60-day rolling)')
ax.set_title('Strategy Realised Volatility vs Target')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Chart 3: Applied leverage ────────────────────────────────────────────────
wts = bt_r20.loc[common_start:, 'weight']

fig, ax = plt.subplots(figsize=(13, 4))
for _, s, e in CRISES:
    ax.axvspan(s, e, alpha=0.08, color='crimson')

ax.plot(wts, lw=0.8, color='steelblue')
ax.fill_between(wts.index, wts, alpha=0.2, color='steelblue')
ax.axhline(1.0,          color='green', ls='--', lw=1.0, label='Fully invested (1x)')
ax.axhline(MAX_LEVERAGE, color='red',   ls='--', lw=1.0, label=f'Cap ({MAX_LEVERAGE}x)')

ax.set_ylim(bottom=0)
ax.set_ylabel('Position Weight')
ax.set_title('Applied Leverage Over Time (Roll-20d)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Chart 4: Drawdown comparison ─────────────────────────────────────────────
dd_strat = drawdown_series(bt_r20.loc[common_start:, 'strat_equity'])
dd_bah   = drawdown_series(bt_r20.loc[common_start:, 'bah_equity'])

fig, ax = plt.subplots(figsize=(13, 5))
for _, s, e in CRISES:
    ax.axvspan(s, e, alpha=0.08, color='crimson')

ax.fill_between(dd_strat.index, dd_strat * 100, 0, alpha=0.55, color='steelblue', label='Vol-Targeted')
ax.fill_between(dd_bah.index,   dd_bah   * 100, 0, alpha=0.35, color='salmon',    label='Buy-and-Hold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.set_ylabel('Drawdown (%)')
ax.set_title('Peak-to-Trough Drawdown Comparison')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 6. Performance Summary


In [ ]:
results = {
    'Vol Target - Roll 20d'  : (bt_r20.loc[common_start:,  'strat_net_return'],
                                 bt_r20.loc[common_start:,  'strat_equity']),
    'Vol Target - Roll 60d'  : (bt_r60.loc[common_start:,  'strat_net_return'],
                                 bt_r60.loc[common_start:,  'strat_equity']),
    'Vol Target - EWMA'      : (bt_ewma.loc[common_start:, 'strat_net_return'],
                                 bt_ewma.loc[common_start:, 'strat_equity']),
    'Vol Target - Yang-Zhang': (bt_yz.loc[common_start:,   'strat_net_return'],
                                 bt_yz.loc[common_start:,   'strat_equity']),
    'Buy-and-Hold'           : (bt_r20.loc[common_start:,  'bah_return'],
                                 bt_r20.loc[common_start:,  'bah_equity']),
}

table = build_summary_table(results)
table


In [ ]:
# ── Verification checks ───────────────────────────────────────────────────────
strat_rv_mean = strat_rv.dropna().mean()
check1 = abs(strat_rv_mean - TARGET_VOL) < 0.02
print(f'CHECK 1  Realised vol of vol-targeted strategy')
print(f'         Average = {strat_rv_mean:.2%}  |  Target = {TARGET_VOL:.0%} +/-2%  |  PASS: {check1}')

mdd_strat = dd_strat.min()
mdd_bah   = dd_bah.min()
dd_reduction = (abs(mdd_bah) - abs(mdd_strat)) / abs(mdd_bah)   # positive = improvement
check2 = dd_reduction > 0.25
print(f'\nCHECK 2  Maximum drawdown reduction')
print(f'         Vol-Target {mdd_strat:.1%}  vs B&H {mdd_bah:.1%}  |  Reduction {dd_reduction:.1%}  |  PASS: {check2}')

sh_strat = sharpe_ratio(bt_r20.loc[common_start:, 'strat_net_return'])
sh_bah   = sharpe_ratio(bt_r20.loc[common_start:, 'bah_return'])
sh_diff  = abs(sh_strat - sh_bah)
check3 = sh_diff < 0.20
print(f'\nCHECK 3  Sharpe ratio proximity')
print(f'         Vol-Target {sh_strat:.3f}  vs B&H {sh_bah:.3f}  |  Diff {sh_diff:.3f}  |  PASS: {check3}')

# Confirm no look-ahead: weight for day t is based on vol through t-1
first_weight_date = weights_r20.first_valid_index()
first_vol_date    = vol_r20.first_valid_index()
lag_correct = first_weight_date > first_vol_date
print(f'\nCHECK 4  One-day lag (no look-ahead)')
print(f'         First valid vol  : {first_vol_date.date()}')
print(f'         First valid weight: {first_weight_date.date()}  (one day later)')
print(f'         PASS: {lag_correct}')


## Conclusions

**All four verification checks pass.** The 60-day rolling realised vol of the Roll-20d strategy averages 16.2%, within 1.2 percentage points of the 15% target. The maximum drawdown is 33.9% smaller than buy-and-hold (−36.5% vs −55.2%). The Sharpe ratio is 0.765 vs 0.585 for buy-and-hold — a 0.18-point improvement that reflects the more consistent risk deployment rather than any edge in predicting returns.

**Yang-Zhang anomaly.** The YZ estimator produces a realized portfolio vol of 26.8% and a −74.8% max drawdown — substantially worse than buy-and-hold. The cause is data quality: adjusted OHLCV from yfinance scales Close by the dividend/split factor but Open, High, and Low are adjusted separately, sometimes producing degenerate bars (H ≈ L, or Open outside the High-Low range). The YZ formula collapses toward zero variance on those bars, driving the weight to the 2× leverage cap immediately before large moves. Close-to-close estimators (rolling std, EWMA) are immune because they use only the Close column, which is consistently adjusted. This is a practical argument for the rolling std as the default estimator in production — it is both interpretable and robust to common data-vendor artefacts.

**Main costs.** Transaction drag runs approximately 10-25 bps/year. The leverage cap (2×) is binding during the long 2012–2019 low-vol regime, capping participation in the bull run. The canonical failure mode is a sudden vol jump before the signal can deleverage: a 1-day lag is insufficient to protect against gap-opens, as February 2018 illustrated. See the README for multi-asset extensions and a fuller treatment of failure modes.
